In [ ]:
# 🩻 ChestNet — ResNet50 Transfer Learning (Full A→Z Pipeline)

**Dataset:** NIH ChestX-ray14 random sample (`sample_labels.csv`, 5,606 images) · **multi-label**
**Model:** ResNet50, ImageNet-pretrained, fully fine-tuned · **Framework:** PyTorch
**Settings:** follow `configs/base.yaml` + `configs/resnet50.yaml` so the comparison with the other models is fair.

| Step | What it does |
|---|---|
| 0 | Install / imports |
| 1 | Config ⚙️ |
| 2 | Helper functions 🧰 |
| 3 | Load + clean labels, quick EDA 🏷️ |
| 4 | Patient-level train/val/test split ✂️ |
| 5 | Resize + cache images (run once) 🖼️ |
| 6 | Dataset, augmentations, DataLoaders 🗂️ |
| 7 | Sanity check ✅ |
| 8 | ResNet50 model 🧠 |
| 9 | Training 🏋️ |
| 10 | Evaluation on the test set 📈 |
| 11 | Single-image prediction + Grad-CAM 🔥 |

> 💡 Run the cells **top to bottom**. Everything is written to `outputs/resnet50/` (outside the `ChestNet` git folder).

In [ ]:
## 0️⃣ Install & imports

In [ ]:
import json
import os
import random
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision import transforms as T

from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             multilabel_confusion_matrix)

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
## 1️⃣ Config ⚙️
Change paths and hyperparameters **here only**.

In [ ]:
# ---------- Paths ----------
from pathlib import Path

# ---------- Paths ----------
ROOT = Path("/content/drive/MyDrive/ChestNet")
DATA_DIR = ROOT / "data"
RAW_IMAGE_DIR = DATA_DIR / "raw_images"          # put images_001 ... images_012 folders here
IMAGE_DIR = DATA_DIR / "images_256"              # resized copies are written here (step 3)
CSV_PATH = DATA_DIR / "Data_Entry_2017.csv"      # NIH metadata file
SPLIT_DIR = DATA_DIR / "splits"                  # train/val/test CSVs (step 4)
STATS_PATH = DATA_DIR / "norm_stats.json"        # mean/std (step 5)
OUT_DIR = ROOT / "outputs" / "ResNet"
DATASET_DIR = IMAGE_DIR            # 📁 images_256 — where your PNGs actually are
MAX_IMAGES_PER_CLASS = 3000        # 🧮 cap for "No Finding" (pick what you like)        # checkpoints, plots, reports
# ---------- Labels (14 NIH diseases) ----------
CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(CLASSES)

# ---------- Image ----------
RESIZE_TO = 256      # size stored on disk (step 3)
IMG_SIZE = 224       # size fed to the network
IN_CHANNELS = 1      # grayscale X-rays

# ---------- Split ----------
TEST_FRAC = 0.20
VAL_FRAC = 0.10
SEED = 42

# ---------- Training ----------
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7               # early-stopping patience (epochs)
MAX_POS_WEIGHT = 50.0      # cap for class-imbalance weights

# ⚠️ In Jupyter on Windows, num_workers > 0 usually crashes (classes defined in a
# notebook can't be sent to worker processes). Keep 0 in the notebook.
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"  # mixed precision only on GPU

OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

In [ ]:
## 2️⃣ Helper functions 🧰

In [ ]:
def set_seed(seed: int = SEED):
    # Same seed -> same split, same augmentations, same initial head weights
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True   # fastest conv algorithm for a fixed image size


def make_labels(df: pd.DataFrame) -> pd.DataFrame:
    # "Effusion|Mass" -> one 0/1 column per class (multi-hot)
    df = df.copy()
    labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = labels.apply(lambda l: int(c in l))
    # "No Finding" must not co-exist with a disease
    disease_cols = [c for c in CLASSES if c != "No Finding"]
    if "No Finding" in CLASSES:
        df.loc[df[disease_cols].sum(axis=1) > 0, "No Finding"] = 0
    return df


def compute_aucs(labels: np.ndarray, probs: np.ndarray) -> np.ndarray:
    # ROC-AUC per class; NaN if a class has only positives or only negatives in this set
    aucs = []
    for i in range(labels.shape[1]):
        if len(np.unique(labels[:, i])) < 2:
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
    return np.array(aucs)


def plot_history(history: pd.DataFrame, save_path=None):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history["epoch"], history["train_loss"], label="train")
    ax[0].plot(history["epoch"], history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(history["epoch"], history["train_auc"], label="train")
    ax[1].plot(history["epoch"], history["val_auc"], label="val")
    ax[1].set_title("Macro AUC"); ax[1].set_xlabel("epoch"); ax[1].legend()
    ax[2].plot(history["epoch"], history["lr"])
    ax[2].set_title("Learning rate"); ax[2].set_xlabel("epoch")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


set_seed()

In [ ]:
## 3️⃣ Load + clean labels, quick EDA 🏷️

In [ ]:
raw = pd.read_csv(CSV_PATH)
print("Rows in CSV:", len(raw))

# Find every PNG under the dataset folder (handles images/, sample/images/, images_001/ ...)
image_paths = {}
for p in DATASET_DIR.rglob("*.png"):
    image_paths.setdefault(p.name, p)          # keep the first copy if a file appears twice
raw["path"] = raw["Image Index"].map(image_paths)
missing = raw["path"].isna().sum()
print(f"Images found on disk: {len(image_paths)} | CSV rows without an image: {missing}")
raw = raw.dropna(subset=["path"]).reset_index(drop=True)

# Age is stored as text like "058Y" -> number of years; impossible ages -> NaN
raw["Age"] = pd.to_numeric(raw["Patient Age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
raw.loc[raw["Age"] > 100, "Age"] = np.nan

df = make_labels(raw)

# Create the "No Finding" column from raw labels if it's not already in CLASSES
if "No Finding" not in df.columns:
    df["No Finding"] = raw["Finding Labels"].eq("No Finding").astype(int).values

# Keep only images that have at least one of our 14 labels or No Finding
valid_cols = [c for c in CLASSES if c in df.columns] + ["No Finding"]
df = df[df[valid_cols].sum(axis=1) > 0].reset_index(drop=True)

# Cap "No Finding" so it does not swamp the diseases
nf = df[df["No Finding"] == 1]
if len(nf) > MAX_IMAGES_PER_CLASS:
    drop_idx = nf.sample(len(nf) - MAX_IMAGES_PER_CLASS, random_state=SEED).index
    df = df.drop(drop_idx).reset_index(drop=True)

print("Images kept:", len(df), "| Patients:", df["Patient ID"].nunique())
counts = df[[c for c in CLASSES if c in df.columns]].sum().sort_values()
print(counts.to_string())

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
counts.plot.barh(ax=ax[0], color="steelblue"); ax[0].set_title("Images per class")
df[[c for c in CLASSES if c in df.columns]].sum(axis=1).value_counts().sort_index().plot.bar(ax=ax[1], color="seagreen")
ax[1].set_title("Labels per image"); ax[1].set_xlabel("# labels")
df["View Position"].value_counts().plot.bar(ax=ax[2], color="salmon")
ax[2].set_title("View position (AP vs PA)")
plt.tight_layout(); plt.savefig(OUT_DIR / "eda_overview.png", dpi=110); plt.show()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

print("Exists?", RAW_IMAGE_DIR.exists())
print("PNG count:", len(list(DATA_DIR.rglob("*.png"))))
for p in list(DATA_DIR.rglob("*.png"))[:3]:
    print(p)

In [ ]:
df = make_labels(raw)

# 🩺 Make the "No Finding" column straight from the CSV text
df["No Finding"] = raw["Finding Labels"].eq("No Finding").astype(int).values

# ✂️ Cap "No Finding" so it does not swamp the diseases
nf = df[df["No Finding"] == 1]
if len(nf) > MAX_IMAGES_PER_CLASS:
    drop_idx = nf.sample(len(nf) - MAX_IMAGES_PER_CLASS, random_state=SEED).index
    df = df.drop(drop_idx).reset_index(drop=True)

print("Images kept:", len(df), "| Patients:", df["Patient ID"].nunique())

In [ ]:
## 4️⃣ Patient-level split ✂️
All images of one patient go into **only one** split → no data leakage. If the team split exists, it is re-used.

In [ ]:
# Fallback configurations if not defined in your main Config cell
VAL_SIZE = globals().get("VAL_FRAC", 0.10)
TEST_SIZE = globals().get("TEST_FRAC", 0.20)
SPLIT_CANDIDATES = 100
MIN_POSITIVES_PER_SPLIT = 2
USE_SHARED_SPLITS = False
SHARED_SPLIT_DIR = DATA_DIR / "shared_splits"

def split_by_patient(df: pd.DataFrame):
    # Positives per class for each patient
    per_patient = df.groupby("Patient ID")[CLASSES].sum()
    patients = per_patient.index.values
    overall = df[CLASSES].mean().values                       # overall prevalence per class
    n_val, n_test = int(len(patients) * VAL_SIZE), int(len(patients) * TEST_SIZE)
    rng = np.random.default_rng(SEED)

    best, best_score = None, np.inf
    for _ in range(SPLIT_CANDIDATES):
        perm = rng.permutation(patients)
        groups = {"test": perm[:n_test], "val": perm[n_test:n_test + n_val], "train": perm[n_test + n_val:]}
        score = 0.0
        for name, pats in groups.items():
            n_imgs = df["Patient ID"].isin(pats).sum()
            pos = per_patient.loc[pats].sum().values
            score += np.abs(pos / n_imgs - overall).sum()       # distance from overall prevalence
            if name != "train" and pos.min() < MIN_POSITIVES_PER_SPLIT:
                score += 10                                     # strong penalty: too few positives
        if score < best_score:
            best, best_score = groups, score

    return {k: df[df["Patient ID"].isin(v)].reset_index(drop=True) for k, v in best.items()}

shared_files = [SHARED_SPLIT_DIR / f"{s}.csv" for s in ("train", "val", "test")]
if USE_SHARED_SPLITS and all(f.exists() for f in shared_files):
    print("Using the team's shared split 🤝")
    splits = {}
    for s, f in zip(("train", "val", "test"), shared_files):
        names = pd.read_csv(f)[["Image Index"]]
        splits[s] = names.merge(make_labels(raw), on="Image Index", how="inner")
else:
    print("Creating a new patient-level split")
    splits = split_by_patient(df)

train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

# 🔒 Leakage check: no patient may appear in two splits
tr, va, te = (set(d["Patient ID"]) for d in (train_df, val_df, test_df))
assert not (tr & va) and not (tr & te) and not (va & te), "Patient leakage between splits!"

for name, d in splits.items():
    d[["Image Index", "Patient ID", "Finding Labels"] + CLASSES].to_csv(SPLIT_DIR / f"{name}.csv", index=False)

summary = pd.DataFrame({n: d[CLASSES].sum() for n, d in splits.items()})
summary.loc["# images"] = [len(d) for d in splits.values()]
summary.loc["# patients"] = [d["Patient ID"].nunique() for d in splits.values()]
print("No patient leakage ✅")
summary

In [ ]:
## 5️⃣ Resize + cache images 🖼️ (run once)
Resizes the 1024×1024 PNGs to 224×224 grayscale once → each epoch is much faster.

In [ ]:
CACHE_DIR = IMAGE_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def cache_one(src: Path):
    dst = CACHE_DIR / src.name
    if dst.exists():
        return
    img = Image.open(src).convert("L")                                  # X-rays are grayscale
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
    img.save(dst)

needed = pd.concat([train_df, val_df, test_df])["path"].map(Path).tolist()
with ThreadPoolExecutor(max_workers=8) as ex:
    list(tqdm(ex.map(cache_one, needed), total=len(needed), desc="caching"))
print("Cached images:", len(list(CACHE_DIR.glob("*.png"))), "✅")

In [ ]:
## 6️⃣ Dataset, augmentations, DataLoaders 🗂️

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class ChestXrayDataset(Dataset):
    # Returns (image_tensor[3,224,224], multi_hot_label[8]). Images are pre-loaded into RAM.

    def __init__(self, df: pd.DataFrame, transform=None):
        self.labels = df[CLASSES].values.astype("float32")
        self.names = df["Image Index"].values
        self.transform = transform
        self.images = [np.array(Image.open(CACHE_DIR / n)) for n in tqdm(self.names, leave=False)]

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])            # grayscale PIL image
        if self.transform:
            img = self.transform(img)
        return img, torch.from_numpy(self.labels[idx])


def get_transforms(train: bool):
    # ⚠️ No horizontal flip: a flipped X-ray puts the heart on the wrong side.
    to_tensor = [
        T.Grayscale(num_output_channels=3),                # ResNet50 was trained on 3-channel images
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
    if train:
        return T.Compose([
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),  # positioning
            T.ColorJitter(brightness=0.1, contrast=0.1),                              # exposure
            *to_tensor,
        ])
    return T.Compose(to_tensor)                            # val/test: no randomness


train_ds = ChestXrayDataset(train_df, get_transforms(train=True))
val_ds = ChestXrayDataset(val_df, get_transforms(train=False))
test_ds = ChestXrayDataset(test_df, get_transforms(train=False))

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, generator=g, **loader_kw)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kw)
print(f"Batches -> train {len(train_loader)}, val {len(val_loader)}, test {len(test_loader)}")

In [ ]:
## 7️⃣ Sanity check ✅
Check shapes, value ranges and labels **before** training.

In [ ]:
x, y = next(iter(train_loader))
print("Image batch:", tuple(x.shape), x.dtype, f"min={x.min():.2f} max={x.max():.2f} mean={x.mean():.2f}")
print("Label batch:", tuple(y.shape), "| positives per class:", y.sum(0).int().tolist())

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = (x[i] * std + mean).clamp(0, 1)[0]                # undo normalization for display
    names = [CLASSES[j] for j in torch.where(y[i] == 1)[0]]
    ax.imshow(img, cmap="gray"); ax.set_title(", ".join(names), fontsize=8); ax.axis("off")
plt.suptitle("Augmented training samples")
plt.tight_layout(); plt.savefig(OUT_DIR / "sample_batch.png", dpi=100); plt.show()